In [1]:
import sys
sys.path.append("../")
import importlib

import torch
import torch.nn.functional as F
from torch_geometric.loader import DataLoader
from torch_geometric.utils import from_networkx

import GNNInterpreter
from GNNInterpreter.gnninterpreter import *


In [2]:
config = {
    "DATASET_NAME": "PROTEINS", 
    "MAX_NODES": 20, 
    "device": "cpu"
}

# Data

In [3]:
import new_src.dataAdapter
from new_src.dataAdapter import load_dataset
importlib.reload(new_src.dataAdapter)

data = load_dataset(config.get("DATASET_NAME"))
cls_split = data.split_by_class()

# Explainee

In [4]:
import new_src.explainee
from new_src.explainee import fit_explainee
importlib.reload(new_src.explainee)

explainee, _ = fit_explainee(
    dataset_name = config.get("DATASET_NAME"),
    root="data",
    hidden=64,
    layers=3,
    dropout=0.2,
    epochs=10,
    batch_size=32,
    lr=1e-3,
    device=config.get("device")
)


# Extract embeddings. 
mean_embeds = [
    torch.cat([explainee(batch.to(config.get("device")))["embeds"]
               for batch in DataLoader(subset, batch_size=64)])
         .mean(dim=0)
    for subset in cls_split
]
mean_embeds = [m.detach() for m in mean_embeds]

[001/10] train_loss=0.6370  val_acc=0.595
[002/10] train_loss=0.5923  val_acc=0.640
[003/10] train_loss=0.5888  val_acc=0.649
[004/10] train_loss=0.5844  val_acc=0.730
[005/10] train_loss=0.5570  val_acc=0.712
[006/10] train_loss=0.5547  val_acc=0.775
[007/10] train_loss=0.5440  val_acc=0.775
[008/10] train_loss=0.5375  val_acc=0.766
[009/10] train_loss=0.5472  val_acc=0.739
[010/10] train_loss=0.5525  val_acc=0.730
Test Accuracy: 0.786


# GED Approximator

# Generator

In [5]:
import new_src.graph_level_dist
importlib.reload(new_src.graph_level_dist)

from new_src.graph_level_dist import dummyDist

In [6]:
def train_generator(cls_idx, data):
    trainer = Trainer(
        sampler=(
            s := GraphSampler(
                    max_nodes=config.get("MAX_NODES"),
                    num_node_cls=len(data.NODE_CLS),
                    num_edge_cls=len(data.EDGE_CLS), 
                    temperature=0.15,
                    learn_node_feat=len(data.NODE_CLS) > 0, 
                    learn_edge_feat=len(data.EDGE_CLS) > 0, 
            )
        ),
        discriminator=explainee,
        criterion=WeightedCriterion([
            dict(key="logits", criterion=ClassScoreCriterion(class_idx=cls_idx, mode='maximize'), weight=1),
            dict(key="embeds", criterion=EmbeddingCriterion(target_embedding=mean_embeds[cls_idx]), weight=50),
            dict(key="cont_data", criterion=dummyDist(), weight=1), 
        ]),
        optimizer=(o := torch.optim.SGD(s.parameters(), lr=1)),
        scheduler=torch.optim.lr_scheduler.ExponentialLR(o, gamma=1),
        dataset=data,
        budget_penalty=BudgetPenalty(budget=10, order=2, beta=1),
    )

    trainer.train(
        iterations=200,
        target_probs={cls_idx: (0.9, 1.0)},
        target_size=30,
        w_budget_init=0.5,
        w_budget_inc=1.1,
        w_budget_dec=0.95,
        k_samples=32
    )

    example = trainer.evaluate(threshold=0.5)

    example = from_networkx(example)

    if "label" in example: 
        example.x = F.one_hot(
            example.label, num_classes=len(data.NODE_CLS)
        ).float()

    if "edge_label" in example: 
        example.edge_attr = F.one_hot(
            example.edge_label, num_classes=len(data.EDGE_CLS)
        ).float()

    example.y = torch.tensor(cls_idx).float()
    
    return example

In [7]:
import new_src.utils
from new_src.utils import plot_graph
importlib.reload(new_src.utils)



sample = train_generator(0, data), 
plot_graph(sample[0], data)

  0%|          | 0/200 [00:00<?, ?it/s]

DataBatch(x=[640, 3], edge_index=[2, 12160], edge_weight=[12160], batch=[640], ptr=[33])
DataBatch(x=[640, 3], edge_index=[2, 12160], edge_weight=[12160], batch=[640], ptr=[33])
DataBatch(x=[640, 3], edge_index=[2, 12160], edge_weight=[12160], batch=[640], ptr=[33])
DataBatch(x=[640, 3], edge_index=[2, 12160], edge_weight=[12160], batch=[640], ptr=[33])
DataBatch(x=[640, 3], edge_index=[2, 12160], edge_weight=[12160], batch=[640], ptr=[33])
DataBatch(x=[640, 3], edge_index=[2, 12160], edge_weight=[12160], batch=[640], ptr=[33])
DataBatch(x=[640, 3], edge_index=[2, 12160], edge_weight=[12160], batch=[640], ptr=[33])
DataBatch(x=[640, 3], edge_index=[2, 12160], edge_weight=[12160], batch=[640], ptr=[33])
DataBatch(x=[640, 3], edge_index=[2, 12160], edge_weight=[12160], batch=[640], ptr=[33])
DataBatch(x=[640, 3], edge_index=[2, 12160], edge_weight=[12160], batch=[640], ptr=[33])
DataBatch(x=[640, 3], edge_index=[2, 12160], edge_weight=[12160], batch=[640], ptr=[33])
DataBatch(x=[640, 3],

KeyboardInterrupt: 

# Evaluation